In [ ]:
# === STEP 0: VERSION CONFIRMATION (edit this, then Run All) ===
# Muse's clearance message says the current version is: "phase0-preprocess-v3"
# Copy it EXACTLY below. The notebook refuses to run on mismatch.
CONFIRM_VERSION = "TYPE-VERSION-HERE"
# ==============================================================

NOTEBOOK_VERSION = "phase0-preprocess-v3"  # embedded by Muse — do not edit
NOTEBOOK_CHANGELOG = "v3: study-range builds (3x1469; /kaggle/working 20GB cap), savez_compressed shards, disk guard, output-size probe gate; cache v4"

if CONFIRM_VERSION != NOTEBOOK_VERSION:
    raise SystemExit(
        "VERSION MISMATCH — not cleared to run.\n"
        f"Embedded: {NOTEBOOK_VERSION} / You typed: {CONFIRM_VERSION}\n"
        "Stop. Check Muse's message for the correct version ID."
    )
print(f"Version verified: {NOTEBOOK_VERSION}")
print(f"Changelog: {NOTEBOOK_CHANGELOG}")


# Phase 0 — DICOM → uint8 cache (the pixel foundation)

Deterministic preprocessing for the RSNA Knee Abnormality Detection competition.
Reads ~4,407 knee-MRI studies (~819k DICOMs), applies the consensus recipe
(geometric slice sort, per-series 1–99 percentile window, mm/px resample,
130 mm crop, R→L canonicalization, uint8), selects one fluid-sensitive series
per plane, and writes sharded uint8 cache + manifest.

**Order of operations:** version gate → startup/config → DICOM schema audit →
tiny-study smoke → throughput probe → full build → verification.
Nothing proceeds past a failed gate. Phase 0 needs no GPU — run on CPU to save quota.


In [ ]:
import os, sys, json, time, hashlib, re, traceback
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

print("python:", sys.version.split()[0])
for _pkg in ["numpy", "pydicom", "PIL", "pandas", "matplotlib", "scipy", "torch"]:
    try:
        _m = __import__(_pkg)
        print(f"  {_pkg}: {getattr(_m, '__version__', 'ok')}")
    except ImportError:
        print(f"  {_pkg}: MISSING")
try:
    import pylibjpeg  # noqa: F401
    print("  pylibjpeg: ok (JPEG transfer syntaxes decodable)")
except ImportError:
    print("  pylibjpeg: MISSING — JPEG-compressed DICOMs will fail loudly at decode")

# GPU info (informational only — Phase 0 is CPU-bound)
try:
    import torch
    print("  torch cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("  cuda device:", torch.cuda.get_device_name(0))
except Exception as _e:
    print("  torch check failed:", _e)
print("NOTE: Phase 0 needs no GPU. Run this notebook CPU-only to save GPU quota.")

# Resample backend is byte-affecting -> resolved once, encoded in CACHE_VERSION.
try:
    import cv2  # noqa: F401
    RESAMPLE_BACKEND = "cv2"
except ImportError:
    RESAMPLE_BACKEND = "scipy"
print("resample backend:", RESAMPLE_BACKEND)


def print_input_tree(base="/kaggle/input", max_depth=2):
    base = Path(base)
    if not base.is_dir():
        print(f"{base}: not present")
        return
    for root, dirs, files in os.walk(base):
        depth = len(Path(root).relative_to(base).parts)
        if depth > max_depth:
            dirs[:] = []
            continue
        print(f"{'  ' * depth}{Path(root).name}/ [{len(dirs)} dirs, {len(files)} files]")


def find_data_root(base="/kaggle/input", max_depth=6):
    """Locate the competition data root = directory containing train.csv.

    Searches deep (depth >= 4): Kaggle changed the mount layout mid-competition
    (type-prefixed nested mounts), so the path is NEVER hardcoded.
    """
    base = Path(base)
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        rel = Path(root).relative_to(base)
        if len(rel.parts) > max_depth:
            dirs[:] = []
            continue
        if "train.csv" in files:
            hits.append(Path(root))
    if not hits:
        return None
    for h in hits:  # prefer the hit that names the competition
        if "rsna" in str(h).lower() or "knee" in str(h).lower():
            print(f"train.csv candidates: {len(hits)} -> chose {h}")
            return h
    print(f"train.csv candidates: {len(hits)} -> chose {hits[0]} (no rsna/knee hint)")
    return hits[0]


print_input_tree()
if Path("/kaggle/input").is_dir():
    DATA_ROOT = find_data_root("/kaggle/input")
else:
    print("not on Kaggle; trying ./data")
    DATA_ROOT = Path("./data") if (Path("./data") / "train.csv").exists() else None

if DATA_ROOT is None:
    raise SystemExit("FATAL: could not locate train.csv. Attach the competition data and re-run.")
DATA_ROOT = Path(DATA_ROOT)
TRAIN_CSV = DATA_ROOT / "train.csv"
TRAIN_SERIES_DIR = DATA_ROOT / "train_series"
print("DATA_ROOT:", DATA_ROOT)
print("train_series present:", TRAIN_SERIES_DIR.is_dir())
if not TRAIN_SERIES_DIR.is_dir():
    raise SystemExit(f"FATAL: {TRAIN_SERIES_DIR} not found under DATA_ROOT.")

# ---- every byte-affecting knob lives in CONFIG; all are encoded in CACHE_VERSION ----
CONFIG = {
    "target_px": 336,                  # output H=W after crop
    "crop_mm": 130.0,                  # anatomical crop, centered on image center
    "pct_lo": 1.0,                     # per-series percentile window (never global)
    "pct_hi": 99.0,
    "resample_backend": RESAMPLE_BACKEND,  # "cv2" or "scipy" — different bytes!
    "resample_order": 1,               # bilinear (scipy path; cv2 uses INTER_AREA/LINEAR)
    "mirror_policy": "none_tag_absent",   # ImageLaterality absent in 100% of series (audit 2026-09-30); no canonicalization, never guess
    "sort": "geometric_IPP_IOP",       # NEVER filename
    "photometric": "invert_MONOCHROME1",
    "rescale": "apply_slope_intercept",
    "series_selection": "fs_preferred_fallback_best",  # FS preferred per plane; best-available fallback (never drop a study for missing FS)
    "pad_value": 0,
    "output_dtype": "uint8",
    "studies_per_shard": 100,          # ~1.0 GB/shard at 3 series x 30 slices x 336^2
    "num_workers": 4,
}
CONFIG["spacing_mm"] = CONFIG["crop_mm"] / CONFIG["target_px"]  # 130/336 ~= 0.3869 mm/px

CACHE_VERSION = (
    f"p0_px{CONFIG['target_px']}"
    f"_crop{int(CONFIG['crop_mm'])}mm"
    f"_p{int(CONFIG['pct_lo']):02d}-{int(CONFIG['pct_hi']):02d}"
    f"_rs{CONFIG['resample_backend']}"
    f"_latnone"      # ImageLaterality tag absent 100% -> no canonicalization
    f"_geosort"
    f"_fsfb"         # fs-preferred series selection with best-available fallback
    f"_u8_v4"        # v4: compressed shard container (decoded bytes identical to v3)
)
print("\nCONFIG:")
print(json.dumps(CONFIG, indent=2))
print("CACHE_VERSION:", CACHE_VERSION)

# --- build range: /kaggle/working persists only 20 GB; the full 4407-study cache is
# ~52 GB, so the build is split into 3 runs of 1469 studies -> 3 Kaggle Datasets.
# Run A: (0, 1469)   Run B: (1469, 2938)   Run C: (2938, 4407)
STUDY_START = 0      # inclusive
STUDY_END   = 1469   # exclusive
OUT_DIR = Path("/kaggle/working") / f"cache_{CACHE_VERSION}_s{STUDY_START:04d}-{STUDY_END:04d}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

# train.csv MUST be read with latin-1: reports are multilingual (EN/ES/FR/DE/...)
# with accented characters that pandas' default UTF-8 decoder mangles. Reports also
# contain embedded newlines (~58k physical lines for 4,407 rows) -> real CSV parser.
train_df = pd.read_csv(TRAIN_CSV, encoding="latin-1", engine="python")
print(f"\ntrain.csv: {len(train_df)} rows, columns={list(train_df.columns)}")
if "StudyInstanceUID" not in train_df.columns:
    raise SystemExit("FATAL: train.csv has no StudyInstanceUID column.")
csv_studies = set(train_df["StudyInstanceUID"].astype(str))
disk_studies = sorted([d.name for d in TRAIN_SERIES_DIR.iterdir() if d.is_dir()])
print(f"studies in train.csv: {len(csv_studies)} | study dirs on disk: {len(disk_studies)}")
if not disk_studies:
    raise SystemExit(f"FATAL: no study directories under {TRAIN_SERIES_DIR}.")
_only_csv = sorted(csv_studies - set(disk_studies))
_only_disk = sorted(set(disk_studies) - csv_studies)
if _only_csv:
    print(f"WARNING: {len(_only_csv)} studies in CSV but not on disk, e.g. {_only_csv[:3]}")
if _only_disk:
    print(f"WARNING: {len(_only_disk)} studies on disk but not in CSV, e.g. {_only_disk[:3]}")
# The DICOMs on disk are the ground truth for a cache build; the CSV is a cross-check.
STUDY_DIRS = [TRAIN_SERIES_DIR / s for s in disk_studies]
assert 0 <= STUDY_START < STUDY_END <= len(STUDY_DIRS), \
    f"bad range [{STUDY_START}, {STUDY_END}) for {len(STUDY_DIRS)} studies"
STUDY_DIRS = STUDY_DIRS[STUDY_START:STUDY_END]
print(f"studies in train.csv: {len(csv_studies)} | study dirs on disk: {len(disk_studies)}")
print(f"THIS RUN range: [{STUDY_START}, {STUDY_END}) -> {len(STUDY_DIRS)} studies to process")

# NOTE (fold grouping — Phase 1+ concern, NOT Phase 0's job; flagged so it isn't forgotten):
# 49 report texts are shared across 183 studies. Studies sharing a report share a target
# vector, so folds must group on report text as well as scanner fingerprint.


## 1. DICOM schema audit (before locking any preprocessing decision)

Reads headers only (no pixels) from ~50 studies spread across the dataset.
Fail-fast gates: geometric sorting is load-bearing — if IPP/IOP are missing in
>5% of series we stop here instead of building on a broken assumption.


In [ ]:
import pydicom

AUDIT_TAGS = [
    "ImagePositionPatient", "ImageOrientationPatient", "ImageLaterality",
    "PhotometricInterpretation", "RescaleSlope", "RescaleIntercept",
    "ScanningSequence", "SeriesDescription", "RepetitionTime", "EchoTime",
    "Manufacturer", "ManufacturerModelName", "SoftwareVersions",
    "MagneticFieldStrength", "ImagingFrequency", "Anatomical_Plane",
    "PixelSpacing", "SliceThickness", "InstanceNumber",
    "SeriesInstanceUID", "SOPInstanceUID",
]

N_AUDIT = 50
_step = max(1, len(STUDY_DIRS) // N_AUDIT)
audit_dirs = STUDY_DIRS[::_step][:N_AUDIT]
print(f"auditing {len(audit_dirs)} studies (every {_step}th of {len(STUDY_DIRS)})")

presence = {t: 0 for t in AUDIT_TAGS}
n_series = 0
laterality_counts, photometric_counts, plane_counts = {}, {}, {}
desc_tokens = {}
header_failures = 0

for sdir in audit_dirs:
    for serdir in sorted([d for d in sdir.iterdir() if d.is_dir()]):
        dcm_files = sorted([f for f in serdir.iterdir()
                            if f.is_file() and f.suffix.lower() == ".dcm"])
        if not dcm_files:
            continue
        try:
            ds = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True)
        except Exception:
            header_failures += 1
            continue
        n_series += 1
        for t in AUDIT_TAGS:
            if getattr(ds, t, None) not in (None, ""):
                presence[t] += 1
        _lat = str(getattr(ds, "ImageLaterality", "") or "").strip().upper()
        _lkey = _lat if _lat in ("L", "R") else "(other/empty)"
        laterality_counts[_lkey] = laterality_counts.get(_lkey, 0) + 1
        _pm = str(getattr(ds, "PhotometricInterpretation", "") or "(missing)")
        photometric_counts[_pm] = photometric_counts.get(_pm, 0) + 1
        _pl = str(getattr(ds, "Anatomical_Plane", "") or "").strip().lower()
        _pkey = _pl if _pl else "(missing)"
        plane_counts[_pkey] = plane_counts.get(_pkey, 0) + 1
        _desc = str(getattr(ds, "SeriesDescription", "") or "")
        for _tok in re.split(r"[^a-zA-Z0-9]+", _desc.lower()):
            if _tok:
                desc_tokens[_tok] = desc_tokens.get(_tok, 0) + 1

print(f"\nseries audited: {n_series} (header read failures: {header_failures})")
print("\ntag presence (% of series):")
for t in AUDIT_TAGS:
    print(f"  {t:28s} {100.0 * presence[t] / max(n_series, 1):6.2f}%")
print("\nImageLaterality:", laterality_counts)
print("PhotometricInterpretation:", photometric_counts)
print("Anatomical_Plane:", plane_counts)
print("\ntop SeriesDescription tokens (validates fluid-sensitive keywords):")
for _tok, _c in sorted(desc_tokens.items(), key=lambda kv: -kv[1])[:40]:
    print(f"  {_tok:22s} {_c}")

# ---- fail-fast gates ----
AUDIT_PASSED = True
_ipp = presence["ImagePositionPatient"] / max(n_series, 1)
_iop = presence["ImageOrientationPatient"] / max(n_series, 1)
if _ipp < 0.95 or _iop < 0.95:
    AUDIT_PASSED = False
    raise SystemExit(
        f"FATAL: geometric slice sorting impossible — IPP present {100 * _ipp:.1f}%, "
        f"IOP present {100 * _iop:.1f}% (need >=95%). "
        "Do NOT proceed: filename sorting silently destroys 2.5D triplets.")
_lat_missing = laterality_counts.get("(other/empty)", 0) / max(n_series, 1)
if _lat_missing > 0.10:
    print(f"\nWARNING: ImageLaterality missing/empty in {100 * _lat_missing:.1f}% of series — "
          "R->L canonicalization skips those studies (logged per-study, never guessed).")
_n_lat = laterality_counts.get("L", 0) + laterality_counts.get("R", 0)
if _n_lat and max(laterality_counts.get("L", 0), laterality_counts.get("R", 0)) / _n_lat > 0.90:
    print("WARNING: laterality is >90% one-sided — unexpected for knee MRI; verify the tag.")
print("\nAUDIT_PASSED =", AUDIT_PASSED)


## 2. Pipeline library — pure, deterministic functions

No randomness anywhere: every tie-break is a sort on SeriesInstanceUID.
These functions are defined once and used identically by smoke, probe, build,
and verification (train/infer pixel paths must be byte-equal, not just similar).


In [ ]:
# ============================================================
# Phase 0 pipeline library — pure, deterministic functions.
# No randomness: tie-breaks sort on SeriesInstanceUID.
# ============================================================
import numpy as np
import pydicom

# Fluid-sensitive keyword matchers: whole-token (see is_fluid_sensitive).
# NOTE: tokenization, NOT \b regexes: DICOM descriptions use underscores as
# separators ("t2_tse_fs_sag"), and "_" is a regex word char, so \bfs\b would
# MISS "fs" there. Whole-token match also guarantees "fse" (fast spin echo)
# never matches "fs" (fat sat).
FATSAT_RE = re.compile(r"fat[\s\-_]*sat")


def decode_slice(path):
    """DICOM -> (float32 pixels, dataset). RescaleSlope/Intercept applied;
    MONOCHROME1 inverted (high value = black -> flip)."""
    ds = pydicom.dcmread(str(path))
    x = ds.pixel_array.astype(np.float32)
    x = x * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        x = x.max() - x
    return x, ds


def geometric_order(datasets):
    """Sort slices by IPP projected onto the slice normal. NEVER filename.

    (Filename order has measured Spearman rho = -0.012 vs true order.)
    Returns (ordered_datasets, unit_normal_or_None, used_fallback).
    Fallback chain: InstanceNumber, then SOPInstanceUID. The caller must log
    loudly when used_fallback is True — never silent.
    """
    keyed = []
    for ds in datasets:
        try:
            iop = [float(v) for v in ds.ImageOrientationPatient]
            ipp = [float(v) for v in ds.ImagePositionPatient]
            if len(iop) != 6 or len(ipp) != 3:
                raise ValueError("bad IOP/IPP length")
            row = np.array(iop[0:3]); col = np.array(iop[3:6])
            normal = np.cross(row, col)
            n = float(np.linalg.norm(normal))
            if n < 1e-6:
                raise ValueError("degenerate IOP")
            normal = normal / n
            keyed.append((float(np.dot(np.array(ipp), normal)), ds))
        except Exception:
            keyed = None
            break
    if keyed is None:
        def _inum(ds):
            try:
                return int(ds.InstanceNumber)
            except Exception:
                return 0
        ordered = sorted(datasets,
                         key=lambda ds: (_inum(ds), str(getattr(ds, "SOPInstanceUID", ""))))
        return ordered, None, True
    keyed.sort(key=lambda kv: kv[0])
    ds0 = keyed[0][1]
    iop = [float(v) for v in ds0.ImageOrientationPatient]
    normal = np.cross(np.array(iop[0:3]), np.array(iop[3:6]))
    normal = normal / float(np.linalg.norm(normal))
    return [ds for _, ds in keyed], normal, False


def window_series(vol, pct_lo=1.0, pct_hi=99.0):
    """Per-series percentile clip -> [0,1]. vol: (S,H,W) float32.

    Per-series, never global: intensity maxima span 12.7x across series, so a
    global window crushes some series to black and saturates others.
    """
    lo, hi = np.percentile(vol, [pct_lo, pct_hi])
    if not (np.isfinite(lo) and np.isfinite(hi)) or hi <= lo:
        return np.zeros_like(vol), False
    vol = np.clip(vol, lo, hi)
    return (vol - lo) / (hi - lo), True


def resample_slice(img, orig_spacing, target_spacing):
    """img: (H,W) float32; orig_spacing: (row_mm, col_mm) from PixelSpacing."""
    sh, sw = img.shape
    nh = max(1, int(round(sh * orig_spacing[0] / target_spacing)))
    nw = max(1, int(round(sw * orig_spacing[1] / target_spacing)))
    if (nh, nw) == (sh, sw):
        return img
    if CONFIG["resample_backend"] == "cv2":
        import cv2
        interp = cv2.INTER_AREA if (nh < sh or nw < sw) else cv2.INTER_LINEAR
        return cv2.resize(img, (nw, nh), interpolation=interp).astype(np.float32)
    from scipy.ndimage import zoom as _zoom
    return _zoom(img, (nh / sh, nw / sw), order=CONFIG["resample_order"]).astype(np.float32)


def center_crop_pad(img, out_px, pad_value=0):
    """Center-crop to out_px x out_px; symmetric zero-pad if smaller. Deterministic."""
    h, w = img.shape
    ph = max(0, out_px - h); pw = max(0, out_px - w)
    if ph or pw:
        img = np.pad(img, ((ph // 2, ph - ph // 2), (pw // 2, pw - pw // 2)),
                     mode="constant", constant_values=pad_value)
        h, w = img.shape
    sh = (h - out_px) // 2; sw = (w - out_px) // 2
    return img[sh:sh + out_px, sw:sw + out_px]


def plane_from_iop(iop):
    """Fallback plane derivation (host Anatomical_Plane is trustworthy; defensive only)."""
    row = np.array(iop[0:3]); col = np.array(iop[3:6])
    normal = np.cross(row, col)
    n = float(np.linalg.norm(normal))
    if n < 1e-6:
        return "unknown"
    normal = normal / n
    ax = int(np.argmax(np.abs(normal)))
    if abs(normal[ax]) < 0.8:
        return "unknown"
    return ["sagittal", "coronal", "axial"][ax]


def is_fluid_sensitive(ds):
    """Recover fluid-sensitive contrast from headers -> (bool, reason).

    The host's Fluid_Sensitive/Fat_Suppression flags are degenerate in train
    (only (1,1)/(0,0) occur) and may diverge in test -> NOT used.
    Priority: gradient-echo exclusion -> description keywords -> TR/TE.
    """
    desc = str(getattr(ds, "SeriesDescription", "") or "")
    seq = str(getattr(ds, "ScanningSequence", "") or "")
    try:
        te = float(getattr(ds, "EchoTime", 0) or 0)
    except Exception:
        te = 0.0
    try:
        tr = float(getattr(ds, "RepetitionTime", 0) or 0)
    except Exception:
        tr = 0.0
    dl = desc.lower()
    # Whole-token match on tokens split at any non-alphanumeric (underscores ARE
    # separators in DICOM descriptions). "t2_tse_fs_sag" -> {t2,tse,fs,sag};
    # "fse" never equals "fs".
    toks = set(t for t in re.split(r"[^a-z0-9]+", dl) if t)
    seq_vals = [s.strip().upper() for s in seq.split("\\") if s.strip()]
    if "GR" in seq_vals:
        return False, "seq:GRE"  # short TR by design breaks the TR/TE rule
    fs_hit = sorted(toks & {"fs", "stir", "pd", "t2"})
    if FATSAT_RE.search(dl):
        fs_hit.append("fatsat")
    fs_hit.sort()
    if fs_hit:
        return True, "desc:" + "+".join(fs_hit)
    if "t1" in toks:
        return False, "desc:t1"
    if te >= 40:
        return True, "trte:TE>=40"
    if 0 < te < 20 and 0 < tr < 800:
        return False, "trte:T1-like"
    return False, "unknown->not-FS"


def canonicalize_laterality(vol, iop, laterality, normal):
    """Mirror R knees to canonical left orientation; canonicalize sagittal stack
    order for both knees. Patient-X based (DICOM +X = patient's left).

    - coronal/axial (|n.X| small), R only: flip the in-plane axis most aligned
      with patient X. (True image mirror: X -> -X.)
    - sagittal (|n.X| ~ 1): X-mirroring leaves in-plane (Y-Z) pixels untouched,
      so canonicalization == a consistent anatomical stack order. Both L and R
      are forced to lateral->medial (lateral->medial runs along -X for L knees,
      +X for R knees). R-only handling would leave L stacks in whatever order
      the acquisition used — not canonical.
    - L/unknown laterality on coronal/axial: no-op. Unknown laterality: never guess.
    Returns (vol, changed: bool, note: str).

    LATERALITY NOTE (verified 2026-09-30): the R->L mirror is CANONICALIZATION,
    not augmentation — the anatomical medial meniscus lands on the canonical
    medial side, so NO Medial/Lateral label swap is needed. Random horizontal
    flips stay OFF in all training phases because the cache is already
    canonicalized.
    """
    X = np.array([1.0, 0.0, 0.0])
    if normal is None or laterality not in ("L", "R"):
        return vol, False, "no-mirror"
    normal = np.asarray(normal, dtype=np.float64)
    nx = float(np.dot(normal, X))
    if abs(nx) > 0.5:
        # lateral->medial == lat_dir * X, with lat_dir = -1 for L, +1 for R.
        # Slices ascend along +normal: already lateral->medial iff lat_dir*nx > 0.
        lat_dir = -1.0 if laterality == "L" else 1.0
        if lat_dir * nx <= 0:
            return vol[::-1].copy(), True, f"sagittal-{laterality}: reordered to lateral->medial"
        return vol, False, f"sagittal-{laterality}: already lateral->medial"
    if laterality != "R":
        return vol, False, "no-mirror"
    row = np.array(iop[0:3]); col = np.array(iop[3:6])
    axis = 1 if abs(float(np.dot(row, X))) >= abs(float(np.dot(col, X))) else 2
    return np.flip(vol, axis=axis).copy(), True, f"inplane-R: flipped axis {axis}"


def process_series(series_dir):
    """Full per-series pipeline -> (vol (S,P,P) float32 in [0,1], meta dict)."""
    files = sorted([f for f in Path(series_dir).iterdir()
                    if f.is_file() and f.suffix.lower() == ".dcm"])
    if not files:
        raise ValueError("no DICOM files in series dir")
    pairs = []
    for f in files:
        x, ds = decode_slice(f)  # raises on corrupt -> loud series failure
        pairs.append((ds, x))
    dss = [ds for ds, _ in pairs]
    ordered, normal, used_fallback = geometric_order(dss)
    arr_by_id = {id(ds): x for ds, x in pairs}
    vol = np.stack([arr_by_id[id(ds)] for ds in ordered]).astype(np.float32)
    n_in = vol.shape[0]
    if not np.all(np.isfinite(vol)):
        raise ValueError("non-finite pixels after decode")
    warnings = []
    if not used_fallback:
        keys = [round(float(np.dot(
            np.array([float(v) for v in ds.ImagePositionPatient]), normal)), 3)
            for ds in ordered]
        if len(set(keys)) < len(keys):
            warnings.append("duplicate slice positions (overlapping slices)")
    else:
        warnings.append("geometric sort unavailable; InstanceNumber fallback used")
    vol, win_ok = window_series(vol, CONFIG["pct_lo"], CONFIG["pct_hi"])
    if not win_ok:
        warnings.append("degenerate intensities (p99<=p1); zeros emitted")
    if not (np.all(vol >= 0) and np.all(vol <= 1)):
        raise ValueError("windowing produced out-of-range values")
    ds0 = ordered[0]
    try:
        ps = ds0.PixelSpacing
        spacing = (float(ps[0]), float(ps[1]))
    except Exception:
        raise ValueError("PixelSpacing missing or unparseable")
    out = np.stack([resample_slice(s, spacing, CONFIG["spacing_mm"]) for s in vol])
    out = np.stack([center_crop_pad(s, CONFIG["target_px"], CONFIG["pad_value"]) for s in out])
    assert out.shape[1] == CONFIG["target_px"] and out.shape[2] == CONFIG["target_px"]
    assert out.shape[0] == n_in, "slice count changed during processing"
    try:
        iop = [float(v) for v in ds0.ImageOrientationPatient]
    except Exception:
        raise ValueError("ImageOrientationPatient missing")
    plane_tag = str(getattr(ds0, "Anatomical_Plane", "") or "").strip().lower()
    if plane_tag in ("sagittal", "coronal", "axial"):
        plane = plane_tag
    else:
        # Tag absent (common — it may be host CSV metadata, not a DICOM tag):
        # derive geometrically and stay quiet. Warn only on unexpected VALUES.
        plane = plane_from_iop(iop)
        if plane_tag:
            warnings.append(f"Anatomical_Plane='{plane_tag}' unexpected; derived '{plane}'")
    fs, fs_reason = is_fluid_sensitive(ds0)
    meta = {
        "series_uid": str(getattr(ds0, "SeriesInstanceUID", Path(series_dir).name)),
        "series_dir": Path(series_dir).name,
        "plane": plane,
        "is_fs": bool(fs),
        "fs_reason": fs_reason,
        "n_slices": int(out.shape[0]),
        "laterality": str(getattr(ds0, "ImageLaterality", "") or "").strip().upper(),
        "iop": iop,
        "normal": None if normal is None else [float(v) for v in normal],
        "warnings": warnings,
    }
    return out, meta


def select_series(series_infos):
    """One series per plane: fluid-sensitive PREFERRED, best-available FALLBACK.

    Dropping a study for missing FS signal is far worse than caching a
    suboptimal (e.g. T1) series for one plane: the labels still describe anatomy,
    and the manifest records per-plane 'selection' so Phase 2 can account for it.
    Deterministic tie-break: most slices, then SeriesInstanceUID.
    Returns {plane: (vol, meta, selection)} with selection in {'fs','fallback'}.
    """
    by_plane_fs, by_plane_all = {}, {}
    for vol, meta in series_infos:
        if meta["plane"] not in ("sagittal", "coronal", "axial"):
            continue
        by_plane_all.setdefault(meta["plane"], []).append((vol, meta))
        if meta["is_fs"]:
            by_plane_fs.setdefault(meta["plane"], []).append((vol, meta))
    selected = {}
    for plane in ("sagittal", "coronal", "axial"):
        cands = by_plane_fs.get(plane, [])
        selection = "fs"
        if not cands:
            cands = by_plane_all.get(plane, [])
            selection = "fallback"
        if not cands:
            continue
        cands.sort(key=lambda vm: (-vm[1]["n_slices"], vm[1]["series_uid"]))
        vol, meta = cands[0]
        selected[plane] = (vol, meta, selection)
    return selected


def process_study(study_dir):
    """Full study pipeline -> (planes: {plane: uint8 (S,P,P)}, study_meta dict)."""
    study_dir = Path(study_dir)
    series_dirs = sorted([d for d in study_dir.iterdir() if d.is_dir()])
    if not series_dirs:
        raise ValueError("no series directories")
    infos, series_errors = [], []
    for sdir in series_dirs:
        try:
            infos.append(process_series(sdir))
        except Exception as e:
            series_errors.append({"series_dir": sdir.name, "error": str(e)[:300]})
    if not infos:
        raise ValueError(f"all {len(series_dirs)} series failed: {series_errors}")
    lats = [m["laterality"] for _, m in infos if m["laterality"] in ("L", "R")]
    # sorted() before max(): deterministic tie-break (no hash-order dependence)
    laterality = max(sorted(set(lats)), key=lats.count) if lats else "U"
    selected = select_series(infos)
    if not selected:
        raise ValueError("no usable series in any plane (all failed or unclassifiable)")
    planes_out, planes_meta = {}, {}
    for plane, (vol, meta, selection) in selected.items():
        normal = None if meta["normal"] is None else np.array(meta["normal"])
        vol_m, mirrored, note = canonicalize_laterality(vol, meta["iop"], laterality, normal)
        u8 = np.clip(vol_m, 0.0, 1.0)
        u8 = (u8 * 255.0 + 0.5).astype(np.uint8)  # round-half-up, deterministic
        planes_out[plane] = u8
        planes_meta[plane] = {
            "series_uid": meta["series_uid"],
            "series_dir": meta["series_dir"],
            "shape": [int(d) for d in u8.shape],
            "sha256": hashlib.sha256(u8.tobytes()).hexdigest(),
            "n_slices": meta["n_slices"],
            "laterality": laterality,
            "mirrored": bool(mirrored),
            "mirror_note": note,
            "fs_reason": meta["fs_reason"],
            "selection": selection,  # 'fs' or 'fallback' (best-available)
            "series_warnings": meta["warnings"],
        }
    study_meta = {
        "study_uid": study_dir.name,
        "laterality": laterality,
        "planes": planes_meta,
        "series_errors": series_errors,
        "n_series_total": len(series_dirs),
        "n_series_ok": len(infos),
    }
    return planes_out, study_meta


print("pipeline library loaded: decode_slice, geometric_order, window_series,",
      "resample_slice, center_crop_pad, plane_from_iop, is_fluid_sensitive,",
      "canonicalize_laterality, process_series, select_series, process_study")


## 3. Tiny-study smoke (5 studies, full pipeline, eyes on pixels)

Runs the entire pipeline on 5 studies and renders first/middle/last slices per
plane to PNG. **Look at the images** — inverted contrast, wrong orientation, or
garbage windowing is visible in seconds and costs nothing to catch here.


In [ ]:
assert AUDIT_PASSED, "audit must pass before the smoke test"

SMOKE_STUDIES = STUDY_DIRS[:5]
print("smoke studies:", [d.name for d in SMOKE_STUDIES])

t0 = time.time()
for sdir in SMOKE_STUDIES:
    t1 = time.time()
    planes, smeta = process_study(sdir)
    dt = time.time() - t1
    print(f"\n{sdir.name}: {dt:.1f}s | laterality={smeta['laterality']} "
          f"| series ok {smeta['n_series_ok']}/{smeta['n_series_total']} "
          f"| planes={sorted(planes)}")
    for plane, u8 in planes.items():
        pm = smeta["planes"][plane]
        assert u8.dtype == np.uint8, f"dtype {u8.dtype}"
        assert u8.shape[1:] == (CONFIG["target_px"], CONFIG["target_px"]), f"shape {u8.shape}"
        assert u8.shape[0] == pm["n_slices"] and pm["n_slices"] > 0
        print(f"  {plane:9s} shape={str(u8.shape):16s} sha={pm['sha256'][:12]} "
              f"mirror=[{pm['mirror_note']}] fs=[{pm['fs_reason']}]")
        for w in pm["series_warnings"]:
            print(f"    series warning: {w}")
    for se in smeta["series_errors"]:
        print(f"    series skipped: {se['series_dir']}: {se['error']}")
print(f"\nsmoke wall: {time.time() - t0:.1f}s")
SMOKE_PASSED = True

# ---- visual inspection: first / middle / last slice per plane ----
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

PNG_DIR = Path("/kaggle/working") / "phase0_smoke"
PNG_DIR.mkdir(parents=True, exist_ok=True)
for sdir in SMOKE_STUDIES:
    planes, smeta = process_study(sdir)  # re-run: also a light determinism check
    for plane, u8 in planes.items():
        mid = u8.shape[0] // 2
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        for ax, idx, ttl in zip(axes, [0, mid, u8.shape[0] - 1],
                                ["first slice", "middle slice", "last slice"]):
            ax.imshow(u8[idx], cmap="gray", vmin=0, vmax=255)
            ax.set_title(f"{ttl} (idx {idx})")
            ax.axis("off")
        fig.suptitle(f"{sdir.name} | {plane} | shape={u8.shape} | "
                     f"{smeta['planes'][plane]['mirror_note']}")
        p = PNG_DIR / f"{sdir.name}_{plane}.png"
        fig.savefig(p, dpi=80, bbox_inches="tight")
        plt.close(fig)
        display(Image(str(p)))
print("smoke PNGs written to", PNG_DIR)
print("SMOKE_PASSED =", SMOKE_PASSED)


## 4. Throughput probe (20 fresh studies → projected full-build time)

**Gate:** if the projected full-corpus build exceeds 6 hours, we stop here and
revise the chunking/parallelism strategy instead of launching a doomed build.


In [ ]:
assert SMOKE_PASSED, "smoke must pass before the probe"
from concurrent.futures import ThreadPoolExecutor

PROBE_N = min(20, max(1, len(STUDY_DIRS) - 5))
PROBE_STUDIES = STUDY_DIRS[5:5 + PROBE_N]
print(f"probing {len(PROBE_STUDIES)} studies with {CONFIG['num_workers']} workers...")


def _timed(sdir):
    t1 = time.time()
    try:
        _planes, _ = process_study(sdir)
        _nb = sum(v.nbytes for v in _planes.values())
        return (sdir.name, time.time() - t1, None, _nb)
    except Exception as e:
        return (sdir.name, time.time() - t1, str(e)[:200], 0)


t0 = time.time()
with ThreadPoolExecutor(max_workers=CONFIG["num_workers"]) as ex:
    _results = list(ex.map(_timed, PROBE_STUDIES))
wall = time.time() - t0
_ok = [r for r in _results if r[2] is None]
_failed = [r for r in _results if r[2] is not None]
per_study = wall / max(len(PROBE_STUDIES), 1)
total_proj_s = per_study * len(STUDY_DIRS)
print(f"probe: {len(_ok)} ok, {len(_failed)} failed, wall {wall:.1f}s")
print(f"per-study: {per_study:.2f}s  ->  projected full build "
      f"({len(STUDY_DIRS)} studies): {total_proj_s / 3600:.2f}h")
for _name, _dt, _err, _nb in _failed:
    print("  FAILED:", _name, "-", _err)

PROBE_PASSED = False
if _failed:
    raise SystemExit(f"FATAL: {len(_failed)}/{len(_results)} probe studies failed. "
                     "Fix process_study before building.")
per_study_B = float(np.mean([r[3] for r in _ok]))
proj_GB = per_study_B * len(STUDY_DIRS) / 1e9
print(f"projected uncompressed output: {proj_GB:.1f} GB for {len(STUDY_DIRS)} studies "
      "(shards are deflate-compressed, actual will be smaller)")
if proj_GB > 19.0:
    raise SystemExit(f"FATAL: projected {proj_GB:.1f} GB exceeds the ~20 GB /kaggle/working "
                     "cap. Shrink STUDY_END - STUDY_START and re-run.")
if total_proj_s > 6 * 3600:
    raise SystemExit(f"FATAL: projected build {total_proj_s / 3600:.1f}h exceeds the 6h gate. "
                     "Revise chunking/parallelism (see EXECUTION_PROTOCOL) before proceeding.")
PROBE_PASSED = True
print("PROBE_PASSED =", PROBE_PASSED)


## 5. Full cache build (resumable, sharded, failure-tolerant)

- Studies already in `manifest.json` are skipped (resume after interruption).
- Shards are written atomically (tmp + rename); a study only enters the manifest
  when its shard is safely on disk — a crashed partial shard is rebuilt, never trusted.
- One study failing never kills the build: it is logged to `failures.json`.
- Progress + ETA every 50 studies; manifest checkpointed alongside.


In [ ]:
assert PROBE_PASSED, "probe gate must pass before the full build"

MANIFEST_PATH = OUT_DIR / "manifest.json"
manifest = {
    "cache_version": CACHE_VERSION,
    "config": CONFIG,
    "created": datetime.now(timezone.utc).isoformat(),
    "studies": {},
    "failures": [],
}
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text())
    if manifest.get("cache_version") != CACHE_VERSION:
        raise SystemExit(
            f"FATAL: existing manifest cache_version={manifest.get('cache_version')} "
            f"!= {CACHE_VERSION}. Refusing to mix versions — move it aside first.")
    print(f"resuming: {len(manifest['studies'])} studies already cached, "
          f"{len(manifest['failures'])} recorded failures")


def save_manifest():
    tmp = MANIFEST_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(manifest))
    os.replace(tmp, MANIFEST_PATH)  # atomic


todo = [d for d in STUDY_DIRS if d.name not in manifest["studies"]]
print(f"to process: {len(todo)} / {len(STUDY_DIRS)} studies")

_shard_files = sorted(OUT_DIR.glob("shard_*.npz"))
shard_idx = max([int(p.stem.split("_")[1]) for p in _shard_files], default=-1) + 1
shard_buf, shard_pending, shard_members = {}, {}, []


def flush_shard():
    global shard_idx
    if not shard_members:
        return
    name = f"shard_{shard_idx:04d}.npz"
    tmp = OUT_DIR / (name + ".tmp")
    # NOTE: np.savez appends ".npz" to str/Path filenames, so pass an open binary
    # handle to control the exact tmp name (otherwise the atomic rename breaks).
    import shutil as _sh
    if _sh.disk_usage(OUT_DIR).free < 3e9:
        raise RuntimeError(f"disk low ({_sh.disk_usage(OUT_DIR).free / 1e9:.1f} GB free) - "
                           "stopping before shard write to avoid a corrupt shard")
    with open(tmp, "wb") as _f:
        np.savez_compressed(_f, **shard_buf)
    _gb_written[0] += (OUT_DIR / name).stat().st_size / 1e9
    os.replace(tmp, OUT_DIR / name)  # atomic
    for suid in shard_members:
        entry = shard_pending.pop(suid)
        entry["shard"] = name
        manifest["studies"][suid] = entry
    save_manifest()
    print(f"  wrote {name}: {len(shard_members)} studies "
          f"({(OUT_DIR / name).stat().st_size / 1e9:.2f} GB, {_gb_written[0]:.1f} GB total)")
    shard_buf.clear(); shard_members.clear()
    shard_idx += 1


def _process_one(sdir):
    t1 = time.time()
    try:
        planes, smeta = process_study(sdir)
        return (sdir.name, planes, smeta, time.time() - t1, None)
    except Exception:
        return (sdir.name, None, None, time.time() - t1, traceback.format_exc(limit=5))


from concurrent.futures import ThreadPoolExecutor
t_start = time.time()
_done = 0
_gb_written = [0.0]
try:
    with ThreadPoolExecutor(max_workers=CONFIG["num_workers"]) as ex:
        for _name, _planes, _smeta, _dt, _err in ex.map(_process_one, todo):
            _done += 1
            if _err is None:
                for _plane, _u8 in _planes.items():
                    shard_buf[f"{_name}/{_plane}"] = _u8
                shard_pending[_name] = {
                    "shard": None,  # filled at flush
                    "planes": _smeta["planes"],
                    "laterality": _smeta["laterality"],
                    "n_series_total": _smeta["n_series_total"],
                    "n_series_ok": _smeta["n_series_ok"],
                    "series_errors": _smeta["series_errors"],
                }
                shard_members.append(_name)
                if len(shard_members) >= CONFIG["studies_per_shard"]:
                    flush_shard()
            else:
                manifest["failures"].append({"study_uid": _name, "error": _err})
                print(f"  STUDY FAILED [{_done}/{len(todo)}]: {_name}\n{_err}\n")
            if _done % 50 == 0:
                _el = time.time() - t_start
                _rate = _done / max(_el, 1e-6)
                _eta_h = (len(todo) - _done) / _rate / 3600
                print(f"[{_done}/{len(todo)}] {_el / 60:.1f} min elapsed | "
                      f"{_rate:.2f} studies/s | ETA {_eta_h:.1f}h | "
                      f"failures={len(manifest['failures'])}")
                save_manifest()
finally:
    flush_shard()
    # Dedupe failures by study (a study that failed before is retried on resume —
    # transient failures recover; keep only the latest record per study).
    _seen, _dedup = set(), []
    for _f in reversed(manifest["failures"]):
        if _f["study_uid"] not in _seen:
            _seen.add(_f["study_uid"])
            _dedup.append(_f)
    manifest["failures"] = list(reversed(_dedup))
    save_manifest()
    (OUT_DIR / "failures.json").write_text(json.dumps(manifest["failures"], indent=2))

_wall_h = (time.time() - t_start) / 3600
print(f"\nBUILD DONE: cached={len(manifest['studies'])} failed={len(manifest['failures'])} "
      f"wall={_wall_h:.2f}h version={CACHE_VERSION}")


## 6. Verification (determinism, integrity, publish metadata)

- (a) Re-runs the 5 smoke studies through the pipeline and byte-compares against
  the cache — the train/infer pixel path must be **identical**, not just similar.
- (b) Recomputes every sha256 in the manifest from the shards.
- (c) Cache summary. (d) Kaggle Dataset metadata for publishing.


In [ ]:
manifest = json.loads((OUT_DIR / "manifest.json").read_text())
assert manifest["cache_version"] == CACHE_VERSION, "manifest version mismatch"
print(f"manifest: {len(manifest['studies'])} studies, "
      f"{len(manifest['failures'])} failures, version={manifest['cache_version']}")

_shard_cache = {}


def _get_cached(suid, plane):
    shard = manifest["studies"][suid]["shard"]
    assert shard, f"{suid} has no shard recorded"
    if shard not in _shard_cache:
        _shard_cache[shard] = np.load(OUT_DIR / shard)
    return _shard_cache[shard][f"{suid}/{plane}"]


# (a) determinism: byte-compare re-run vs cache on the smoke studies
for sdir in SMOKE_STUDIES:
    _planes, _ = process_study(sdir)
    for _plane, _u8 in _planes.items():
        _cached = _get_cached(sdir.name, _plane)
        assert _cached.shape == _u8.shape and _cached.dtype == _u8.dtype, \
            f"shape/dtype drift: {sdir.name}/{_plane}"
        assert np.array_equal(_cached, _u8), f"NON-DETERMINISTIC: {sdir.name}/{_plane}"
        print(f"  deterministic OK: {sdir.name}/{_plane}")
print("(a) determinism: PASS")

# (b) manifest integrity: verify every checksum from the shards
_by_shard = {}
for _suid, _entry in manifest["studies"].items():
    _by_shard.setdefault(_entry["shard"], []).append(_suid)
_n_checked = 0
for _shard_name in sorted(_by_shard):
    _zf = np.load(OUT_DIR / _shard_name)
    for _suid in _by_shard[_shard_name]:
        for _plane, _pm in manifest["studies"][_suid]["planes"].items():
            _arr = _zf[f"{_suid}/{_plane}"]
            assert _arr.dtype == np.uint8, f"dtype drift: {_suid}/{_plane}"
            _h = hashlib.sha256(_arr.tobytes()).hexdigest()
            assert _h == _pm["sha256"], f"checksum mismatch: {_suid}/{_plane}"
            _n_checked += 1
    _zf.close()
print(f"(b) checksum integrity: PASS ({_n_checked} arrays)")

# (c) summary
_total_slices = sum(_pm["n_slices"] for _e in manifest["studies"].values()
                    for _pm in _e["planes"].values())
_total_bytes = sum(p.stat().st_size for p in OUT_DIR.glob("shard_*.npz"))
_n_planes = {}
for _e in manifest["studies"].values():
    for _pl in _e["planes"]:
        _n_planes[_pl] = _n_planes.get(_pl, 0) + 1
print(f"(c) cache summary: studies={len(manifest['studies'])} "
      f"failures={len(manifest['failures'])} arrays={_n_checked} "
      f"slices={_total_slices} size={_total_bytes / 1e9:.2f} GB")
print(f"    planes cached: {_n_planes} | version={CACHE_VERSION}")
assert _total_bytes / 1e9 < 20.0, \
    f"output {_total_bytes / 1e9:.1f} GB exceeds the 20 GB /kaggle/working cap"
_n_fallback = sum(1 for _e in manifest["studies"].values()
                  for _pm in _e["planes"].values() if _pm.get("selection") == "fallback")
print(f"    fallback selections (non-FS series): {_n_fallback} plane-images — "
      "Phase 2 should know these exist")

# (d) publish metadata — copy the printed JSON to dataset-metadata.json next to
# shard_*.npz + manifest.json + failures.json, then: kaggle datasets create -p <dir>
_ds_meta = {
    "title": f"rsna-knee-phase0-cache-{CACHE_VERSION}-s{STUDY_START:04d}-{STUDY_END:04d}",
    "id": "YOUR_KAGGLE_USERNAME/rsna-knee-phase0-cache-" f"s{STUDY_START:04d}-{STUDY_END:04d}",
    "licenses": [{"name": "CC0-1.0"}],
}
print("(d) dataset-metadata.json (replace YOUR_KAGGLE_USERNAME):")
print(json.dumps(_ds_meta, indent=2))
print("\nFiles to publish:", sorted(p.name for p in OUT_DIR.iterdir()))
print("\nPHASE 0 COMPLETE — cache is deterministic, checksummed, and versioned.")


## Done

**Artifacts in `/kaggle/working/cache_<CACHE_VERSION>_s<START>-<END>/` (one range per run):**
`shard_*.npz` (uint8, keys `<study_uid>/<plane>`), `manifest.json`
(study → shard, sha256, series list, warnings, cache version), `failures.json`.

**Next:** publish as a Kaggle Dataset, attach it to the Phase 1 notebook, and log
this run in `docs/run-log.md` (probe numbers, wall time, failures).
